In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scripts.TPS import ThinPlateSpline
from scripts.plotting import *

# ---------- Load vector field ----------
def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values
    return X, V, time

# ---------- File organization ----------
path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv"
}

# Custom layout
row1_names = ["straight_line", "sine_curve", "branch_2", "branch_4"]
row2_names = ["rotation", "spiral", "saddle", "quadratic_source_sink"]
plot_order = row1_names + row2_names

# ---------- 2×4 grid plot ----------
fig, axs = plt.subplots(2, 4, figsize=(20, 10))

for i, (ax, name) in enumerate(zip(axs.ravel(), plot_order)):
    X, V, time = load_vector_field(path_map[name])
    time = (time - np.min(time)) / (np.max(time) - np.min(time))  # normalize time globally

    tps_vf = ThinPlateSpline(X, n_control_points=100)
    tps_vf.fit(V, dof=15)

    if i == 0:
        stream_density = 0.4
        aspect = 2.5
    elif i == 1:
        stream_density = 0.8
        aspect = 2.0
    elif i < 4:
        stream_density = 0.8
        aspect = 1.5
    else:
        stream_density = 1
        aspect = "equal"

    plot_velocity_streamplot(
        X_2d=X,
        tps_vf=tps_vf,
        grid_density=1.0, 
        stream_density=stream_density,
        scatter_color=time,
        scatter_size=40,
        scatter_alpha=0.5,
        ax=ax,
        title=name.replace("_", " "),
        figsize=(5, 4),
        aspect=aspect,
        vmin=0.0,
        vmax=1.0,
        grid_size=50
    )

plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ------------------------------------------
# simulate noisy data (your block unchanged)
# ------------------------------------------
simulation_results = {}
noise, extra_dim = 0.2, 2
for name, path in path_map.items():
    X_gt, V_gt, time = load_vector_field(path)
    X_noisy = X_gt + np.random.normal(scale=noise, size=X_gt.shape)
    V_noisy = V_gt + np.random.normal(scale=noise, size=V_gt.shape)
    X_dummy = np.random.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
    V_dummy = np.random.normal(scale=noise, size=(V_gt.shape[0], extra_dim))
    X = np.hstack([X_noisy, X_dummy])
    V = np.hstack([V_noisy, V_dummy])
    simulation_results[name] = dict(X=X, V=V, X_gt=X_gt, V_gt=V_gt, true_time=time)

In [ ]:
from scripts.VectorFieldEmbedder import *

embedding_results = {}

for name, data in simulation_results.items():
    X = data["X"]
    V = data["V"]
    time = data["true_time"]
    time = (time - np.min(time)) / (np.max(time) - np.min(time))  # normalize

    emb = VectorFieldEmbedder(
        X, V, use_PCA=False,
        embed_kwargs={"n_neighbors": 30, "min_dist": 0.3}
    )
    emb.initialize_embedding()
    emb.optimize()

    embedding_results[name] = {
        "embedder": emb,
        "time": time
    }

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
from scripts.evaluation import evaluate_embedding_method

fig, axs = plt.subplots(1, 8, figsize=(32, 4))
scores = []

for ax, name in zip(axs, embedding_results.keys()):
    emb = embedding_results[name]["embedder"]
    time = embedding_results[name]["time"]
    X_gt = simulation_results[name]["X_gt"]
    V_gt = simulation_results[name]["V_gt"]
    X_emb = emb.X_emb
    V_emb = emb.tps_vf.predict(X_emb)

    # Custom panel layout
    if name == "straight_line":
        stream_density, aspect = 0.5, 2.5
    elif name == "sine_curve":
        stream_density, aspect = 0.5, 2.0
    elif name in {"branch_2", "branch_4"}:
        stream_density, aspect = 0.6, 1.5
    else:
        stream_density, aspect = 0.6, "equal"

    # Plot
    plot_velocity_streamplot(
        X_2d=X_emb,
        tps_vf=emb.tps_vf,
        grid_density=1.0, 
        stream_density=stream_density,
        scatter_color=time,
        scatter_size=800,
        scatter_alpha=0.1,
        ax=ax,
        title=None,
        aspect=aspect,
        vmin=0.0,
        vmax=1.0,
        arrowsize=3.0,
        cmap="viridis",
        show_axes=False,
        streamline_thickness=4.0,
        grid_size=50
    )

    # Score
    metrics = evaluate_embedding_method(X_gt, X_emb, V_gt, V_emb, k=30)
    metrics["dataset"] = name
    scores.append(metrics)

plt.tight_layout()
plt.show()

# Save scores
df = pd.DataFrame(scores).set_index("dataset")
os.makedirs("./data/8_vf_collection", exist_ok=True)
df.to_csv("./data/8_vf_collection/flowmap.csv")

print("\nFlowMap scores saved to ./data/8_vf_collection/flowmap.csv")
print(df.round(4))

In [ ]:
plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    grid_density=1.0, 
    stream_density=stream_density,
    scatter_color=time,
    scatter_size=800,
    scatter_alpha=0.1,
    title=None,
    aspect=aspect,
    vmin=0.0,
    vmax=1.0,
    arrowsize=3.0,
    cmap="viridis",
    show_axes=False,
    streamline_thickness=4.0,
    grid_size=50
)

In [ ]:
from scripts.vector_field_geometry import *

# 1) detect
fps = find_fixed_points_grid(emb.tps_vf, emb.X_emb,
                             grid_size=120, tol_vec=1, tol_merge=1)

# 2) build the grid once (so Jacobian fits reuse it)
Xg, Vg = compute_velocity_on_grid(emb.X_emb, emb.tps_vf, grid_size=120)

# 3) loop through fixed points
for i, fp in enumerate(fps):
    try:
        J = jacobian_from_grid(emb.tps_vf, fp, Xg, Vg, radius=1)
        label = classify_fixed_point(J)
    except ValueError as e:
        label = f"unreliable ({e})"
    print(f"FP{i}: {np.round(fp,3)}  →  {label}")q1

In [ ]:
for name, res in embedding_results.items():
    emb = res["embedder"]
    print(f"\n===== {name.upper()} =====")

    # --- 1. Find fixed points
    fps = find_fixed_points_grid(
        emb.tps_vf, emb.X_emb,
        grid_size=120,
        tol_vec=1,
        tol_merge=3,
    )

    if fps.shape[0] == 0:
        print("No candidate fixed points found.")
        continue

    # --- 2. Build the grid
    Xg, Vg = compute_velocity_on_grid(
        emb.X_emb, emb.tps_vf,
        grid_size=120,
        margin_ratio=0.1,
    )

    # --- 3. Classify and print
    for i, fp in enumerate(fps):
        try:
            J = jacobian_from_grid(emb.tps_vf, fp, Xg, Vg, radius=2)
            print(eigvals(J))
            label = classify_fixed_point(J)
        except ValueError as e:
            label = f"unreliable ({e})"
        print(f" FP{i}: {np.round(fp, 3)} → {label}")

    # --- 4. Plot embedding + fixed points
    plt.figure(figsize=(4, 4))
    plt.scatter(emb.X_emb[:, 0], emb.X_emb[:, 1], s=3, color='gray', alpha=0.5)
    plt.scatter(fps[:, 0], fps[:, 1], s=60, color='red', label='Fixed Points')
    plt.title(f"{name}")
    plt.axis("equal")
    plt.legend()
    plt.tight_layout()
    plt.show()
